# Food Production Challenge 3 - Baseline Submission

This notebook provides a simple baseline for **Food Production Challenge 3: Demand Forecasting**.

**Goal**: Predict `units_sold_next_week` for each region-SKU combination
**Metric**: Root Mean Squared Error (RMSE) - Lower is better

## Instructions:
1. **Replace API credentials** in the first cell with your team's API key and name
2. **Run all cells** to generate and submit baseline predictions
3. **Check the output** for your submission score

This baseline uses only tabular demand data with a simple Random Forest regressor.


In [ ]:
# 1. Initialize Client and Load Data

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from agentds import BenchmarkClient

# 🔑 REPLACE WITH YOUR CREDENTIALS
client = BenchmarkClient(
    api_key="your-api-key-here",        # Get from your team dashboard
    team_name="your-team-name-here"     # Your exact team name
)

# Load data from PVC paths
print("📂 Loading Food Production Challenge 3 data...")

# Load demand data
train_demand = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/demand_train.csv")
test_demand = pd.read_csv("/home/jovyan/shared/datasets/FoodProduction/demand_test.csv")

print(f"✅ Data loaded:")
print(f"   Train demand: {train_demand.shape}")
print(f"   Test demand: {test_demand.shape}")
print(f"   Train columns: {list(train_demand.columns)}")
print(f"   Test columns: {list(test_demand.columns)}")


In [ ]:
# 2. Tabular-Only Baseline Model and Predictions

# From data inspection - demand columns:
# region_id, sku_id, week, sku_base, price, base_price, promo_flag, feature_display, stockout_prev, seasonal_index, units_sold_next_week (train only)

# Select numeric demand features for baseline (exclude ID columns which are not predictive)
demand_features = ['sku_base', 'price', 'base_price', 'promo_flag', 'feature_display', 'stockout_prev', 'seasonal_index']
print(f"📊 Using demand features: {demand_features}")

# Prepare training data (ensure proper data types like performance testing)
X_train = train_demand[demand_features].fillna(0).astype(float)
y_train = train_demand['units_sold_next_week'].astype(float)  # Target variable

# Prepare test data
X_test = test_demand[demand_features].fillna(0).astype(float)

# Train simple Random Forest baseline
print("🤖 Training Random Forest regressor...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
predictions = model.predict(X_test)

# Create submission file (format: region_id,sku_id,week,units_sold_next_week)
submission_df = pd.DataFrame({
    'region_id': test_demand['region_id'],
    'sku_id': test_demand['sku_id'],
    'week': test_demand['week'],
    'units_sold_next_week': predictions
})

# Save predictions
submission_df.to_csv("foodproduction_challenge3_predictions.csv", index=False)
print(f"✅ Predictions saved: {submission_df.shape[0]} predictions")
print(f"   Preview: {submission_df.head(3)}")
print(f"   Demand range: {predictions.min():.1f} to {predictions.max():.1f} units")


In [ ]:
# 3. Submit Predictions

# Submit predictions to the competition
print("🚀 Submitting predictions...")

try:
    result = client.submit_prediction("FoodProduction", 3, "foodproduction_challenge3_predictions.csv")
    
    if result['success']:
        print("✅ Submission successful!")
        print(f"   📊 Score: {result['score']:.4f}")
        print(f"   📏 Metric: {result['metric_name']}")
        print(f"   ✔️  Validation: {'Passed' if result['validation_passed'] else 'Failed'}")
    else:
        print("❌ Submission failed!")
        print(f"   Error details: {result.get('details', {}).get('validation_errors', 'Unknown error')}")
        
except Exception as e:
    print(f"💥 Submission error: {e}")
    print("🔧 Check your API key and team name are correct!")

print("\n🎯 Next steps:")
print("   1. Try incorporating relevant information outside this table!")
print("   2. You've completed all Food Production challenges!")
